# 02 — Price Behavior EDA

**Goal:** Understand how prices actually move so we can design meaningful features and validate the 5% threshold.

Questions answered here:
- How often do prices change?
- How large are typical price drops?
- Is there a weekly or monthly seasonality pattern?
- Which retailer / category shows the most volatility?
- Is the 5% threshold a reasonable signal for 'meaningful' drops?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
from src.db_utils import get_engine

sns.set_theme(style='whitegrid', palette='muted')
engine = get_engine()

In [ ]:
df = pd.read_sql(text("""
    SELECT ph.id, ph.product_id, ph.price::float AS price,
           ph.observed_at, ph.source,
           p.title, p.brand, p.category, r.name AS retailer
    FROM price_history ph
    JOIN products  p ON ph.product_id = p.id
    JOIN retailers r ON p.retailer_id  = r.id
    ORDER BY ph.product_id, ph.observed_at
"""), engine.connect())
df['observed_at'] = pd.to_datetime(df['observed_at'], utc=True)

# Compute price changes within each product
df = df.sort_values(['product_id', 'observed_at'])
df['prev_price']     = df.groupby('product_id')['price'].shift(1)
df['price_change']   = df['price'] - df['prev_price']
df['price_change_pct'] = df['price_change'] / df['prev_price']
df['is_drop']        = (df['price_change'] < 0).astype(int)
df['is_increase']    = (df['price_change'] > 0).astype(int)
df['day_of_week']    = df['observed_at'].dt.dayofweek  # 0=Mon
df['month']          = df['observed_at'].dt.month

# Subset: rows that represent a price change
df_changes = df.dropna(subset=['prev_price'])
df_drops   = df_changes[df_changes['is_drop'] == 1].copy()
print(f'{len(df_changes)} transitions | {len(df_drops)} drops | {len(df_changes) - len(df_drops)} increases/stables')

## 1. Price change distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Absolute change
df_changes['price_change'].hist(bins=40, ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Price change per observation ($)')
axes[0].set_xlabel('Δ price ($)')

# Percentage change — clipped for readability
clipped = df_changes['price_change_pct'].clip(-0.5, 0.5)
clipped.hist(bins=40, ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--')
axes[1].axvline(-0.05, color='orange', linestyle='--', label='-5% threshold')
axes[1].set_title('% price change per observation (clipped ±50%)')
axes[1].set_xlabel('% change')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Drop size percentiles:')
print(df_drops['price_change_pct'].abs().describe(percentiles=[.25,.5,.75,.90,.95]))

## 2. Is 5% the right threshold?

In [ ]:
thresholds = [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
results = []
for t in thresholds:
    meaningful = (df_drops['price_change_pct'].abs() >= t).sum()
    results.append({'threshold': f'{t:.0%}', 'drops_qualifying': meaningful,
                    'pct_of_all_changes': meaningful / len(df_changes) * 100})
print(pd.DataFrame(results).to_string(index=False))

## 3. Drop frequency by retailer and category

In [ ]:
drop_rate = (
    df_changes.groupby('retailer')
    .agg(total_changes=('is_drop', 'count'), drops=('is_drop', 'sum'))
    .assign(drop_rate=lambda x: x['drops'] / x['total_changes'])
)
print('Drop rate by retailer:')
print(drop_rate)

if df_changes['category'].nunique() > 1:
    drop_rate_cat = (
        df_changes.groupby('category')
        .agg(total=('is_drop', 'count'), drops=('is_drop', 'sum'))
        .assign(drop_rate=lambda x: x['drops'] / x['total'])
    )
    print('\nDrop rate by category:')
    print(drop_rate_cat)

## 4. Weekday and monthly patterns

In [ ]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

weekday_drops = df_drops.groupby('day_of_week').size()
weekday_drops.index = [day_names[i] for i in weekday_drops.index]
weekday_drops.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Price drops by day of week')
axes[0].set_xlabel('')

month_drops = df_drops.groupby('month').size()
month_drops.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Price drops by month')
axes[1].set_xlabel('Month')

plt.tight_layout()
plt.show()

## 5. Price volatility by product

In [ ]:
volatility = (
    df.groupby(['product_id', 'title', 'retailer'])['price']
    .agg(std='std', mean='mean', min='min', max='max', count='count')
    .assign(cv=lambda x: x['std'] / x['mean'],
            range_pct=lambda x: (x['max'] - x['min']) / x['mean'])
    .sort_values('cv', ascending=False)
    .reset_index()
)
print('Top 10 most volatile products (coefficient of variation):')
print(volatility[['title', 'retailer', 'cv', 'range_pct', 'count']].head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
volatility['cv'].hist(bins=20, ax=ax)
ax.set_title('Price volatility (coefficient of variation) per product')
ax.set_xlabel('CV (std / mean)')
plt.tight_layout()
plt.show()

## 6. Sample price timelines (top 4 volatile products)

In [ ]:
top4 = volatility.head(4)['product_id'].tolist()
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

for ax, pid in zip(axes.flat, top4):
    sub = df[df['product_id'] == pid].sort_values('observed_at')
    title = sub['title'].iloc[0][:40]
    ax.plot(sub['observed_at'], sub['price'], marker='o', markersize=3)
    ax.set_title(title, fontsize=9)
    ax.set_ylabel('Price ($)')
    ax.tick_params(axis='x', rotation=30, labelsize=7)

plt.suptitle('Price timelines — top 4 volatile products', y=1.01)
plt.tight_layout()
plt.show()